In [ ]:
import numpy as np
from keras.datasets.mnist import load_data
from keras.layers import Dense,Dropout,LeakyReLU,Reshape,Conv2DTranspose,Conv2D,Flatten
from keras.optimizers import Adam
from keras.models import Sequential
import matplotlib.pyplot as plt
from keras.utils import plot_model

In [ ]:
(trainX,_),(_,_) = load_data()
for i,j in enumerate(np.random.randint(0,trainX.shape[0],49)):
    plt.subplot(7,7,i+1)
    plt.imshow(trainX[j],cmap='gray_r')
    plt.axis('off')
plt.show()

In [ ]:
def define_discriminator(in_shape=(28,28,1)):
    model = Sequential()
    model.add(Conv2D(filters=64,kernel_size=(3,3),strides=(2,2),padding='same',input_shape=in_shape))
    model.add(LeakyReLU(negative_slope=0.2))
    model.add(Dropout(0.4))
    model.add(Conv2D(64,(3,3),strides=(2,2),padding='same'))
    model.add(LeakyReLU(negative_slope=0.2))
    model.add(Dropout(0.4))
    model.add(Flatten())
    model.add(Dense(1,activation='sigmoid'))
    opt = Adam(learning_rate=0.0002,beta_1=0.5)
    model.compile(loss='binary_crossentropy',optimizer=opt,metrics=['accuracy'])
    return model

In [ ]:
def load_real_samples():
    (trainX,_),(_,_)= load_data()
    X =np.expand_dims(trainX,axis=-1)
    X= X.astype('float32')
    X= X/255.0
    return X

In [ ]:
def generate_real_samples(datasets,n_sample):
    img = np.random.randint(0,datasets.shape[0],n_sample)
    X= datasets[img]
    y= np.ones((n_sample,1))
    return X,y

In [ ]:
def define_generator(latent_dims):
    model = Sequential()
    #first foundation of the image
    n_nodes = 128*7*7
    model.add(Dense(n_nodes,input_dim=latent_dims))
    model.add(LeakyReLU(negative_slope=0.2))
    model.add(Reshape((7,7,128)))
    #when it has became the 7X7 image with 128 features lets upscale it to get the correct dimensions
    model.add(Conv2DTranspose(128,(4,4),strides=2,padding='same'))
    model.add(LeakyReLU(negative_slope=0.2))
    model.add(Conv2DTranspose(128,(4,4) , strides=2,padding='same'))
    model.add(LeakyReLU(negative_slope=0.2))
    model.add(Conv2D(1,(7,7),activation='sigmoid',padding='same'))
    return model

In [ ]:
#generating the latent points for getting the images 
def generate_latent_point(latent_dim,n_samples):
    x_input=np.random.randn(latent_dim*n_samples)
    x_input=x_input.reshape(n_samples,latent_dim)
    return x_input

In [ ]:
def generate_fake_samples(g_model,latent_dim,n_sample):
    x_input=generate_latent_point(latent_dim,n_sample)
    X= g_model.predict(x_input)
    y=np.zeros((n_sample,1))
    return X,y

In [ ]:
#combining all them models and now making a GAN
def define_gan(g_model,d_model):
    d_model.trainable = False
    model = Sequential()
    model.add(g_model)
    model.add(d_model)
    model.compile(optimizer=Adam(learning_rate=0.0002,beta_1=0.5),loss='binary_crossentropy')
    return model

In [14]:
def train(gan_model,d_model,g_model,dataset,latent_dim,n_epochs=100,n_batch=256):
    batch_per_epoch= int(dataset.shape[0]/n_batch)
    half_batch = int(n_batch /2)
    #manually running the epochs 
    for i in range(n_epochs):
        #enumenumerate batches over training set 
        for j in range(batch_per_epoch):
            X_real,y_real = generate_real_samples(dataset,half_batch)
            #generating the fake
            X_fake,y_fake = generate_fake_samples(g_model,latent_dim,half_batch)
            #creating a trainign stack means ki ek ke upar ek
            X,y = np.vstack((X_real,X_fake)),np.vstack((y_real,y_fake))
            #calculating the losses of both NN
            d_loss , _ = d_model.train_on_batch(X,y)
            #now its time for generator lets generate the latent points
            X_gan =  generate_latent_point(latent_dim,n_batch)
            #now labelling as manupulative
            y_gan = np.ones((n_batch,1))
            g_loss = gan_model.train_on_batch(X_gan,y_gan)
            print(f'Epochs: {i+1} | Batch per Epoch: {j+1} | D_Loss: {d_loss:.3f} | G_Loss: {g_loss:.3f}')
        if (i+1) % 10 == 0:
            summarize_performance(i, g_model,d_model,dataset,latent_dim)

In [ ]:
def summarize_performance(epoch, g_model, d_model, dataset, latent_dim, n_samples=100):
    X_real, y_real = generate_real_samples(dataset, n_samples) 
    _, acc_real = d_model.evaluate(X_real, y_real, verbose=0)
    
    x_fake, y_fake = generate_fake_samples(g_model, latent_dim, n_samples)
    _, acc_fake = d_model.evaluate(x_fake, y_fake, verbose=0)
    print(f'Epoch: {epoch} | Accuracy Real: {acc_real*100:.2f}% | Accuracy Fake: {acc_fake*100:.2f}%')
    save_plot(x_fake, epoch)
	# save the generator model tile file
    filename = 'generator_model_%03d.h5' % (epoch +1 )
    g_model.save(filename)

In [ ]:
# create and save a plot of generated images (reversed grayscale)
def save_plot(examples, epoch, n=10):
	# plot images
	for i in range(n * n):
		# define subplot
		plt.subplot(n, n, 1 + i)
		# turn off axis
		plt.axis('off')
		# plot raw pixel data
		plt.imshow(examples[i, :, :, 0], cmap='gray_r')
	# save plot to file
	filename = 'generated_plot_e%03d.png' % (epoch+1)
	plt.savefig(filename)
	plt.close()

In [ ]:
# size of the latent space
latent_dim = 100
# create the discriminator
d_model = define_discriminator()
# create the generator
g_model = define_generator(latent_dim)
# create the gan
gan_model = define_gan(g_model, d_model)
# load image data
dataset = load_real_samples()
# train model
train(gan_model, d_model, g_model, dataset, latent_dim)

In [15]:
import tensorflow as tf
print("TF version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices('GPU'))
print("Built with CUDA:", tf.test.is_built_with_cuda())

TF version: 2.21.0
GPU devices: []
Built with CUDA: True


W0000 00:00:1783662217.665046  946467 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
